# 08 · Learn a tiny urban world model

**Question:** Can a learned transition model support multi-step counterfactual rollouts?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra to interpret one-step versus rollout error and explain why a good surrogate is not evidence of causal validity.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## State + action → next state
State is dimensionless heat and traffic; actions are cooling and transit intensity.
A linear model learns the transition from synthetic simulator trajectories. Train/test splits use whole trajectories to avoid temporal leakage.
This illustrates the world-model idea of learning dynamics for imagined rollouts. It is not a pretrained visual world model or a reproduction of Ha & Schmidhuber (2018).


In [ ]:
rng=np.random.default_rng(123); states=[]; actions=[]; targets=[]; groups=[]
for episode in range(50):
    state=rng.uniform(0,2,2)
    for t in range(20):
        action=rng.uniform(0,.6,2)
        nxt=transition(state,action,rng.normal(0,.02,2))
        states.append(state.copy()); actions.append(action); targets.append(nxt); groups.append(episode)
        state=nxt
states=np.array(states); actions=np.array(actions); targets=np.array(targets); groups=np.array(groups)
X=features(states,actions); train=groups<40; test=~train
weights=np.linalg.lstsq(X[train],targets[train],rcond=None)[0]
prediction=X[test]@weights
rmse=float(np.sqrt(np.mean((prediction-targets[test])**2)))
persistence=float(np.sqrt(np.mean((states[test]-targets[test])**2)))
assert rmse<persistence
print('Held-out one-step RMSE:',rmse,'| persistence:',persistence)


In [ ]:
def rollout(action, learned=True, horizon=35):
    s=np.array([1.5,1.5]); history=[s.copy()]
    for _ in range(horizon):
        s=(features(s,action)@weights)[0] if learned else transition(s,action)
        history.append(s.copy())
    return np.array(history)
policy=np.array([.4,.5]); learned=rollout(policy); oracle=rollout(policy,False); base=rollout([0,0])
rollout_rmse=float(np.sqrt(np.mean((learned-oracle)**2)))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for i,ax in enumerate(axes):
    ax.plot(base[:,i],label='Learned baseline'); ax.plot(learned[:,i],label='Learned intervention'); ax.plot(oracle[:,i],'--',label='Simulator intervention')
    ax.set(xlabel='Step',ylabel='Dimensionless state',title=['Heat','Traffic'][i]); ax.legend()
plt.tight_layout(); plt.show()
outside=np.array([1.2,1.2]); print('Out-of-training-support action:',outside,'- no validated forecast')
assert (outside>actions.max(axis=0)).all()
export_json('08_world_model.json',dict(synthetic=True,one_step_rmse=rmse,persistence_rmse=persistence,rollout_rmse=rollout_rmse,training_action_max=.6))


## Teaching lab: Audit a learned surrogate
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 08_world_model.json, the action range and the rollout plot. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Compare one-step error with rollout error, identify action extrapolation, and explain what further evidence a causal policy claim needs.

**Implementation brief — review the runnable extension below:**
> Compare learned and persistence baselines at horizons 1, 10 and 35 on whole held-out trajectories. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** No training/test trajectory overlap; all methods see the same initial states and actions; unsupported actions remain labeled.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Learn nonlinear spatial dynamics and expose extrapolation

Introduce nonlinear congestion and fit an ensemble of linear surrogates using whole training trajectories. Evaluate horizons 1, 10 and 35 on held-out episodes, alongside persistence. Map predicted changes for a grid of synthetic neighborhoods. An out-of-support action is shown separately; ensemble spread is a diagnostic, not calibrated real-world uncertainty.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Learn nonlinear spatial dynamics and expose extrapolation', ['Neighborhood states + actions', 'Nonlinear simulator / ensemble', 'Rollout error + support map', 'Simulator fidelity ≠ causality'])
plt.show()


In [ ]:
def ex_dynamics(state,action):
    state=np.asarray(state); action=np.asarray(action)
    return transition(state,action)+np.stack([.025*state[...,1]**2-.03*state[...,0]*action[...,0],.07*np.tanh(state[...,1]**2)],axis=-1)
rng=np.random.default_rng(212); ex_episodes=[]
for episode in range(40):
    state=rng.uniform(.1,2,2); rows=[]
    for step in range(40):
        action=rng.uniform(0,.6,2); nxt=ex_dynamics(state,action); rows.append((state.copy(),action.copy(),nxt.copy())); state=nxt
    ex_episodes.append(rows)
ex_ensemble=[]
for member in range(12):
    selected=rng.choice(30,30,replace=True); rows=[r for episode in selected for r in ex_episodes[episode]]
    s,a,t=map(np.array,zip(*rows)); ex_ensemble.append(np.linalg.lstsq(features(s,a),t,rcond=None)[0])
ex_models=np.array(ex_ensemble); ex_horizons=[1,10,35]; ex_errors=[]; ex_persist=[]
for horizon in ex_horizons:
    errors=[]; persistence=[]
    for episode in range(30,40):
        start=ex_episodes[episode][0][0]; true=start.copy(); pred=start.copy()
        for step in range(horizon):
            action=ex_episodes[episode][step][1]; true=ex_dynamics(true,action); pred=(features(pred,action)@ex_models.mean(axis=0))[0]
        errors.append(np.mean((pred-true)**2)); persistence.append(np.mean((start-true)**2))
    ex_errors.append(float(np.sqrt(np.mean(errors)))); ex_persist.append(float(np.sqrt(np.mean(persistence))))
ex_x,ex_y=np.meshgrid(np.linspace(100,1900,8),np.linspace(100,1900,8)); ex_states=np.column_stack([.5+ex_y.ravel()/2000,.5+ex_x.ravel()/2000])
ex_action=np.array([.4,.5]); ex_base=ex_dynamics(ex_states,[0,0]); ex_policy=ex_dynamics(ex_states,ex_action)
ex_predictions=np.array([features(ex_states,np.tile(ex_action,(64,1)))@model for model in ex_models])
fig,axes=plt.subplots(1,3,figsize=(14,4))
dots=axes[0].scatter(ex_x,ex_y,c=(ex_policy-ex_base)[:,0],s=80,cmap='coolwarm',vmin=-.15,vmax=.15); map_axes(axes[0],'Synthetic neighborhood heat change'); fig.colorbar(dots,ax=axes[0],label='Dimensionless heat')
axes[1].plot(ex_horizons,ex_errors,marker='o',label='Linear surrogate'); axes[1].plot(ex_horizons,ex_persist,marker='s',label='Persistence'); axes[1].set(xlabel='Rollout horizon',ylabel='RMSE',title='Whole held-out trajectories'); axes[1].legend(fontsize=8)
axes[2].add_patch(plt.Rectangle((0,0),.6,.6,color='#087f8c',alpha=.2)); axes[2].scatter([.4,1.2],[.5,1.2],c=['#087f8c','#c44e52'],s=90)
axes[2].annotate('Supported',(.4,.5)); axes[2].annotate('Extrapolation',(1.2,1.2),ha='right'); axes[2].set(xlim=(0,1.4),ylim=(0,1.4),xlabel='Cooling action',ylabel='Transit action',title='Declared training support')
fig.tight_layout(); extension_figure=fig
print('Mean ensemble predictive spread:',float(ex_predictions.std(axis=0).mean()),'— not calibrated uncertainty')
extension_evidence=dict(horizons=ex_horizons,surrogate_rmse=ex_errors,persistence_rmse=ex_persist,training_action_bounds=[0,.6],proposed_action=[1.2,1.2],training_episodes=list(range(30)),test_episodes=list(range(30,40)))
extension_expected=dict(episode_overlap=0,proposed_action_supported=False)
extension_task='Audit the split and action support. Return episode_overlap and proposed_action_supported. Do not infer causal validity from simulator error.'


In [ ]:
extension_prompt,extension_reference=export_investigation('08',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 08_extension_prompt.json and optionally 08_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `08_extension_prompt.json` and `08_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('08_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
